# Estudo Exploratório e Didático do Problema do Caixeiro Viajante (PCV)
## Meta-heurísticas de Busca Local e Populacional: Têmpera Simulada e Algoritmos Genéticos
**Disciplina:** Sistemas Inteligentes — DAINF / UTFPR  
**Autores:** Mário Cordeiro Júnior (mariojuniors31@gmail.com) e André F. Maccarini (andref.maccarini@gmail.com)  

---

### Visão Geral
Este notebook apresenta a implementação e a análise comparativa de duas meta-heurísticas aplicadas ao Problema do Caixeiro Viajante (PCV): a Têmpera Simulada (busca de trajetória única) e os Algoritmos Genéticos (busca populacional).

O código foi desenvolvido em Python nativo com suporte de NumPy e Matplotlib, sem bibliotecas externas de otimização. Cada operador, função de transição e cálculo de aptidão é demonstrado em uma instância controlada com 5 cidades antes da execução dos experimentos nas instâncias de 20, 50 e 100 cidades.


### Fundamentação de IA: Análise PEAS e Propriedades do Ambiente (Russell & Norvig)
Seguindo a formulação clássica de Russell & Norvig, a definição formal do problema parte da caracterização PEAS (*Performance, Environment, Actuators, Sensors*) e das propriedades do ambiente:

| Elemento PEAS | Descrição no Contexto do PCV |
| :--- | :--- |
| **Performance (Desempenho)** | Minimizar a distância euclidiana total percorrida no ciclo hamiltoniano ($f(P) = \sum c_{i, i+1} + c_{n, 1}$). |
| **Environment (Ambiente)** | Grafo completo ponderado com $n$ cidades em coordenadas euclidianas bidimensionais. |
| **Actuators (Atuadores)** | Operadores de perturbação da rota (troca de posições *swap*, recombinação genética *OX*, mutação). |
| **Sensors (Sensores)** | Função de custo da rota que avalia a variação de custo ($\Delta E$) ou o valor de aptidão (*fitness*). |

#### Propriedades do Ambiente de Tarefa:
1. **Totalmente Observável:** O agente tem acesso à matriz completa de distâncias entre todas as cidades.
2. **Determinístico:** As transições entre rotas (trocas ou recombinações) produzem resultados determinísticos.
3. **Estático:** As coordenadas e distâncias entre cidades não se alteram durante o processo de busca.
4. **Discreto:** O espaço de busca é formado por um conjunto finito de permutações de tamanho $(n-1)! / 2$.
5. **Agente Único:** Apenas uma entidade de busca opera sobre o grafo em cada ensaio.

#### Níveis de Abstração da Modelagem:
- **1. Domínio:** Problemas reais de roteirização e logística de entregas.
- **2. Modelagem:** Formulação do problema como Ciclo Hamiltoniano de custo mínimo em grafo euclidiano completo.
- **3. Representação Computacional:** Permutação de inteiros $P = \langle P_0, P_1, \dots, P_{n-1} \rangle$ e matriz de adjacência pré-calculada.
- **4. Implementação:** Algoritmos de Têmpera Simulada e Algoritmo Genético com sementes pseudoaleatórias fixadas.


---
## Parte 1: Formulação Matemática e o "Problema Brinquedo" (5 Cidades)

Formalmente, o PCV simétrico busca encontrar um ciclo hamiltoniano de menor custo em um grafo completo $G = (V, E)$.
Dado um conjunto de $n$ cidades e uma matriz de distâncias $C = (c_{ij})$, uma rota é representada por um vetor de permutação:
$$P = \langle P(0), P(1), \dots, P(n-1) \rangle$$
O custo total da rota que visita todas as cidades e retorna à cidade inicial $P(0)$ é calculado por:
$$f(P) = \sum_{i=0}^{n-2} c_{P(i), P(i+1)} + c_{P(n-1), P(0)}$$

Para validar visualmente e aritmeticamente cada função, definimos um **exemplo brinquedo com 5 cidades** (A, B, C, D, E) com coordenadas conhecidas no plano cartesiano 2D.


In [ ]:
import math
import random
import heapq
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Definir semente global para que todos os experimentos sejam 100% reprodutíveis
SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)

# Definindo 5 cidades com coordenadas cartesianas controladas
# A=(0,0), B=(0,300), C=(400,300), D=(400,0), E=(200,150)
cidades_brinquedo = [
    (0, 0),       # Cidade 0: A
    (0, 300),     # Cidade 1: B
    (400, 300),   # Cidade 2: C
    (400, 0),     # Cidade 3: D
    (200, 150)    # Cidade 4: E (ponto central)
]
nomes_cidades = ['A', 'B', 'C', 'D', 'E']

def gerar_matriz_distancias(cidades):
    """Calcula a matriz euclidiana de distâncias simétricas entre todas as cidades."""
    n = len(cidades)
    matriz = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            d = math.hypot(cidades[i][0] - cidades[j][0], cidades[i][1] - cidades[j][1])
            matriz[i][j] = d
            matriz[j][i] = d
    return matriz

matriz_brinquedo = gerar_matriz_distancias(cidades_brinquedo)
print("Matriz de Distâncias do Exemplo Brinquedo (5 Cidades):")
for i, linha in enumerate(matriz_brinquedo):
    vals = "  ".join([f"{v:6.1f}" for v in linha])
    print(f"  {nomes_cidades[i]}: [{vals}]")



### Teste Unitário 1: Validação Manual do Custo
Conferência manual da rota canônica: $A \to B \to C \to D \to E \to A$ (índices `0 -> 1 -> 2 -> 3 -> 4 -> 0`):
- $A(0,0) \to B(0,300) = 300$
- $B(0,300) \to C(400,300) = 400$
- $C(400,300) \to D(400,0) = 300$
- $D(400,0) \to E(200,150) = \sqrt{200^2 + 150^2} = 250$
- $E(200,150) \to A(0,0) = \sqrt{200^2 + 150^2} = 250$
- **Custo esperado:** $300 + 400 + 300 + 250 + 250 = 1500.0$.


In [ ]:
def calcular_custo_total(rota, matriz_dist):
    """Calcula a soma das distancias ao longo da rota fechada e o retorno a origem."""
    n = len(rota)
    custo = matriz_dist[rota[-1], rota[0]] # retorno a primeira cidade
    for i in range(n - 1):
        custo += matriz_dist[rota[i], rota[i + 1]]
    return custo

def formatar_rota(rota, nomes):
    """Exibe a rota com os nomes legíveis das cidades."""
    return " -> ".join([nomes[c] for c in rota]) + f" -> {nomes[rota[0]]}"

rota_teste = [0, 1, 2, 3, 4] # A -> B -> C -> D -> E -> A
custo_obtido = calcular_custo_total(rota_teste, matriz_brinquedo)

print(f"Rota de Teste: {formatar_rota(rota_teste, nomes_cidades)}")
print(f"Custo Calculado pelo Algoritmo: {custo_obtido:.2f}")
print(f"Custo Calculado Manualmente:    1500.00")
assert abs(custo_obtido - 1500.0) < 1e-6, "Erro no calculo do custo!"
print(">> [TESTE UNITARIO VALIDADO COM SUCESSO]")



---
## Parte 2: Por que a Busca Sistemática ($A^*$) Falha no PCV?

Antes de aplicar meta-heurísticas, é fundamental que o estudante compreenda: **por que não usamos um algoritmo clássico exato como o $A^*$ ou Busca em Largura?**
No $A^*$, o espaço de busca é formulado como uma **árvore de estados parciais**, onde cada nó na fronteira de prioridade representa um caminho incompleto que ainda precisa ser expandido.
- Para 5 cidades, o $A^*$ consegue encontrar a rota ótima com facilidade.
- Mas observe abaixo quantos nós ele precisa manter na memória (fronteira) apenas para 5 cidades!


In [ ]:
def a_star_tsp(matriz_dist):
    """Busca A* (Uniform Cost Search / Heuristica Admissivel Zero) para o PCV."""
    n = len(matriz_dist)
    # heap de prioridade: (f_score, g_custo, cidade_atual, mascara_visitadas, caminho)
    inicio = (0.0, 0.0, 0, 1 << 0, [0])
    fronteira = [inicio]
    nos_expandidos = 0
    tamanho_max_fronteira = 1
    todos_visitados = (1 << n) - 1
    
    while fronteira:
        f, g, u, mask, path = heapq.heappop(fronteira)
        nos_expandidos += 1
        
        if mask == todos_visitados:
            custo_total = g + matriz_dist[u, 0]
            caminho_completo = path + [0]
            return caminho_completo, custo_total, nos_expandidos, tamanho_max_fronteira
            
        for v in range(n):
            if not (mask & (1 << v)):
                novo_g = g + matriz_dist[u, v]
                nova_mask = mask | (1 << v)
                heapq.heappush(fronteira, (novo_g, novo_g, v, nova_mask, path + [v]))
                
        if len(fronteira) > tamanho_max_fronteira:
            tamanho_max_fronteira = len(fronteira)
            
    return None, float('inf'), nos_expandidos, tamanho_max_fronteira

caminho_otimo_astar, custo_astar, nos_exp, max_front = a_star_tsp(matriz_brinquedo)
print("Resultado da Busca Exata A* no Problema Brinquedo (5 Cidades):")
print(f"  Rota Ótima: {formatar_rota(caminho_otimo_astar[:-1], nomes_cidades)}")
print(f"  Custo Ótimo: {custo_astar:.2f}")
print(f"  Nós Expandidos: {nos_exp}")
print(f"  Tamanho Máximo da Fronteira na Memória: {max_front} nós")



### Comparativo de Complexidade de Memória: Busca Clássica ($A^*$) vs. Busca Local
Diferença de complexidade espacial entre os paradigmas:
- **$A^*$ (Busca Sistemática em Árvore):** A fronteira de prioridade cresce em ordem de $O(b^d)$, armazenando caminhos parciais abertos. Para $n=20$, o número de prefixos possíveis atinge ordem fatorial, levando ao esgotamento da memória RAM.
- **Têmpera Simulada (Busca de Trajetória Única):** Mantém apenas uma rota em memória a cada passo. Complexidade de espaço: $O(n)$.
- **Algoritmo Genético (Busca Populacional):** Mantém uma população de tamanho $P$. Complexidade de espaço: $O(P \cdot n)$.

| Algoritmo | Complexidade de Memória | Escalabilidade ($n=100$ Cidades) | Garantia Teórica |
| :--- | :--- | :--- | :--- |
| **$A^*$ (Busca em Árvore)** | $O(b^d)$ — Exponencial | **Inviável** (esgotamento de RAM para $n \ge 15$) | Ótimo global sob $h$ admissível |
| **Têmpera Simulada (TS)** | $O(n)$ — Linear estrito | **Rápida** (fração de segundo) | Ótimo local com escape estocástico |
| **Algoritmo Genético (AG)** | $O(P \cdot n)$ — Linear na população | **Alta** (segundos) | Heurística populacional competitiva |


---
## Parte 3: Têmpera Simulada (Simulated Annealing) — Passo a Passo

A Têmpera Simulada opera por perturbações locais em uma única solução completa.
Os quatro blocos fundamentais são:
1. **Operador de Vizinhança:** Troca da posição de duas cidades (*swap*);
2. **Avaliação Incremental (Delta):** Cálculo da diferença de custo $\Delta E = f(P_{\text{novo}}) - f(P_{\text{atual}})$;
3. **Critério de Aceitação de Metropolis:** Aceitar sempre se $\Delta E < 0$, ou aceitar com probabilidade $e^{-\Delta E / T}$ se $\Delta E \ge 0$;
4. **Esquema de Resfriamento:** Função que decrementa a temperatura $T$ a cada iteração.


### Teste Unitário 2: Operador de Vizinhança (Swap)
Inspeção da rota antes e após a troca da posição de duas cidades no vetor de permutação.


In [ ]:
def trocar_cidades(rota, i, j):
    """Gera um vizinho trocando as posicoes i e j no vetor de permutacao."""
    nova_rota = list(rota)
    nova_rota[i], nova_rota[j] = nova_rota[j], nova_rota[i]
    return nova_rota

rota_original = [0, 1, 2, 3, 4] # A -> B -> C -> D -> E -> A
rota_vizinha = trocar_cidades(rota_original, 1, 4) # Troca B (indice 1) com E (indice 4)

custo_orig = calcular_custo_total(rota_original, matriz_brinquedo)
custo_viz = calcular_custo_total(rota_vizinha, matriz_brinquedo)
delta_calculado = custo_viz - custo_orig

print(f"Rota Original: {formatar_rota(rota_original, nomes_cidades)} | Custo: {custo_orig:.2f}")
print(f"Rota Vizinha:  {formatar_rota(rota_vizinha, nomes_cidades)} | Custo: {custo_viz:.2f}")
print(f"Delta E (Variação de Custo): {delta_calculado:+.2f}")



### Teste Unitário 3: Critério de Metropolis e Dinâmica da Temperatura
O Critério de Metropolis governa a probabilidade de aceitação de movimentos na Têmpera Simulada:
- Se $\Delta E < 0$: a rota melhora $\implies$ aceita deterministicamente (100%);
- Se $\Delta E \ge 0$: a rota piora $\implies$ aceita com probabilidade $p = e^{-\Delta E / T}$.

Comportamento da probabilidade de aceitação para uma piora de $+100$ unidades em diferentes temperaturas:


In [ ]:
def criterio_metropolis(delta_e, temperatura, rng=None):
    if rng is None:
        rng = random.Random()
    if delta_e < 0:
        return True, 1.0
    if temperatura <= 1e-9:
        return False, 0.0
    probabilidade = math.exp(-delta_e / temperatura)
    aceita = rng.random() < probabilidade
    return aceita, probabilidade

delta_piora = 100.0 # Uma rota candidata que piorou em 100 unidades
print("Simulação do Critério de Metropolis para piora fixa de Delta E = +100:")
print(f"{'Temperatura (T)':<18} | {'Probabilidade (p)':<20} | {'Comportamento do Agente':<30}")
print("-" * 75)

for T in [10000, 1000, 500, 100, 50, 10, 1]:
    _, p = criterio_metropolis(delta_piora, T)
    comportamento = "Exploração livre (quase tudo aceito)" if p > 0.8 else (
        "Equilíbrio exploração / refino" if p > 0.2 else "Refino estrito (quase nada aceito)"
    )
    print(f"T = {T:<14} | p = {p:<18.4f} | {comportamento}")



### Teste Unitário 4: Esquemas de Resfriamento
Avaliação das quatro equações de decaimento térmico $T$ ao longo das iterações $k$:
1. **Linear:** $T_k = T_{k-1} - \alpha$
2. **Geométrico:** $T_k = T_{k-1} \cdot \alpha$
3. **Exponencial:** $T_k = T_{k-1} \cdot e^{-\alpha}$
4. **Logarítmico:** $T_k = \frac{T_0}{1 + \alpha \ln(1 + k)}$


In [ ]:
passos = np.arange(1, 1001)
T0 = 1000.0

# Curvas analiticas de resfriamento
temp_linear = np.maximum(0.01, T0 - passos * 1.0)
temp_geom = T0 * (0.995 ** passos)
temp_expo = T0 * np.exp(-0.005 * passos)
temp_log = T0 / (1.0 + 0.05 * np.log(1.0 + passos))

plt.figure(figsize=(10, 4.2), dpi=150)
plt.plot(passos, temp_linear, label='Linear (taxa=1.0)', color='tab:red', linestyle='--')
plt.plot(passos, temp_geom, label='Geométrico (alpha=0.995)', color='tab:blue', linewidth=2)
plt.plot(passos, temp_expo, label='Exponencial (alpha=0.005)', color='tab:green')
plt.plot(passos, temp_log, label='Logarítmico (alpha=0.05)', color='tab:purple')

plt.title('Comparação dos 4 Esquemas de Resfriamento da Temperatura (Têmpera Simulada)', fontsize=11, fontweight='bold')
plt.xlabel('Passo de Iteração (k)', fontsize=10)
plt.ylabel('Temperatura (T)', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()



### Algoritmo Completo: Têmpera Simulada com Armazenamento da Melhor Rota
Implementação combinando o operador swap, cálculo incremental de custo em $O(1)$ e registro da melhor solução visitada ao longo da busca (*best-so-far*).


In [ ]:
def delta_swap(rota, i, j, matriz_dist):
    """Calcula a diferenca de custo delta da troca de duas cidades em O(1)."""
    n = len(rota)
    if i == j: return 0.0
    if i > j: i, j = j, i
    i_prev, i_next = (i - 1) % n, (i + 1) % n
    j_prev, j_next = (j - 1) % n, (j + 1) % n
    if j == i + 1 or (i == 0 and j == n - 1):
        if i == 0 and j == n - 1:
            u, v = j, i
            u_prev, v_next = (u - 1) % n, (v + 1) % n
            return (matriz_dist[rota[u_prev], rota[v]] + matriz_dist[rota[u], rota[v_next]] -
                    matriz_dist[rota[u_prev], rota[u]] - matriz_dist[rota[v], rota[v_next]])
        else:
            return (matriz_dist[rota[i_prev], rota[j]] + matriz_dist[rota[i], rota[j_next]] -
                    matriz_dist[rota[i_prev], rota[i]] - matriz_dist[rota[j], rota[j_next]])
    else:
        return (matriz_dist[rota[i_prev], rota[j]] + matriz_dist[rota[j], rota[i_next]] +
                matriz_dist[rota[j_prev], rota[i]] + matriz_dist[rota[i], rota[j_next]] -
                matriz_dist[rota[i_prev], rota[i]] - matriz_dist[rota[i], rota[i_next]] -
                matriz_dist[rota[j_prev], rota[j]] - matriz_dist[rota[j], rota[j_next]])

def tempera_simulada(cidades, matriz_dist, temp_inicial=1000.0, temp_min=0.01, taxa=0.99, esquema='geometrico', seed=42):
    rng = random.Random(seed)
    n = len(cidades)
    
    # 1. Estado inicial aleatorio
    rota_atual = list(range(n))
    rng.shuffle(rota_atual)
    custo_atual = calcular_custo_total(rota_atual, matriz_dist)
    
    melhor_rota = list(rota_atual)
    melhor_custo = custo_atual
    
    T = float(temp_inicial)
    k = 1
    historico_custos = [melhor_custo]
    
    while T > temp_min and k < 10000:
        # 2. Gerar vizinho swap com delta O(1)
        i, j = rng.sample(range(n), 2)
        delta_e = delta_swap(rota_atual, i, j, matriz_dist)
        
        # 3. Criterio de Metropolis
        aceita, _ = criterio_metropolis(delta_e, T, rng)
        if aceita:
            rota_atual[i], rota_atual[j] = rota_atual[j], rota_atual[i]
            custo_atual += delta_e
            if custo_atual < melhor_custo:
                melhor_custo = custo_atual
                melhor_rota = list(rota_atual)
                
        # 4. Resfriamento
        if esquema == 'geometrico':
            T *= taxa
        elif esquema == 'linear':
            T -= taxa
        elif esquema == 'exponencial':
            T *= math.exp(-taxa)
        elif esquema == 'logaritmico':
            T = temp_inicial / (1.0 + taxa * math.log(1.0 + k))
            
        k += 1
        historico_custos.append(melhor_custo)
        
    return melhor_rota, melhor_custo, historico_custos

# Executando no exemplo brinquedo de 5 cidades
melhor_rota_ts, melhor_custo_ts, hist_ts = tempera_simulada(
    cidades_brinquedo, matriz_brinquedo, temp_inicial=500.0, temp_min=0.01, taxa=0.98, seed=42
)
print("Resultado da Têmpera Simulada no Problema Brinquedo:")
print(f"  Melhor Rota Encontrada: {formatar_rota(melhor_rota_ts, nomes_cidades)}")
print(f"  Custo Mínimo: {melhor_custo_ts:.2f}")
print(f"  Total de Iterações Realizadas: {len(hist_ts)}")



---
## Parte 4: Algoritmo Genético (AG) — Passo a Passo

O Algoritmo Genético opera sobre uma população de permutações candidatas. Seus componentes fundamentais são:
1. **Cromossomo e População:** Cada indivíduo é uma rota completa; a aptidão é dada pelo inverso do custo ($1 / \text{custo}$).\n2. **Seleção por Torneio:** Sorteiam-se $k$ indivíduos aleatoriamente e o de menor custo é selecionado.\n3. **Cruzamento Ordenado (Ordered Crossover - OX):** Recombina dois progenitores preservando a validade do ciclo hamiltoniano, sem repetições nem omissões de cidades.\n4. **Mutação Swap e Elitismo:** Troca pontual de duas posições para manter variabilidade e preservação do melhor indivíduo histórico a cada geração.\n

### Teste Unitário 5: População Inicial e Cálculo de Aptidão (Fitness)


In [ ]:
def criar_populacao_inicial(tam_pop, num_cidades, rng):
    populacao = []
    for _ in range(tam_pop):
        ind = list(range(num_cidades))
        rng.shuffle(ind)
        populacao.append(ind)
    return populacao

def calcular_aptidao(rota, matriz_dist):
    custo = calcular_custo_total(rota, matriz_dist)
    return 1.0 / custo if custo > 0 else 1e9

rng_teste = random.Random(42)
pop_teste = criar_populacao_inicial(tam_pop=4, num_cidades=5, rng=rng_teste)

print("Amostra de População Inicial (4 Indivíduos no Exemplo Brinquedo):")
for i, ind in enumerate(pop_teste):
    c = calcular_custo_total(ind, matriz_brinquedo)
    fit = calcular_aptidao(ind, matriz_brinquedo)
    print(f"  Indivíduo {i+1}: {formatar_rota(ind, nomes_cidades)} | Custo: {c:6.1f} | Aptidão: {fit:.6f}")



### Teste Unitário 6: Seleção por Torneio
Sorteiam-se $k=3$ concorrentes da população e seleciona-se o indivíduo de menor custo (maior aptidão).


In [ ]:
def selecao_torneio(populacao, matriz_dist, k=3, rng=None):
    if rng is None:
        rng = random.Random()
    amostra_indices = rng.sample(range(len(populacao)), k)
    melhor_idx = min(amostra_indices, key=lambda idx: calcular_custo_total(populacao[idx], matriz_dist))
    return populacao[melhor_idx]

pai_selecionado = selecao_torneio(pop_teste, matriz_brinquedo, k=3, rng=rng_teste)
print(f"Vencedor do Torneio: {formatar_rota(pai_selecionado, nomes_cidades)} (Custo: {calcular_custo_total(pai_selecionado, matriz_brinquedo):.2f})")



### Teste Unitário 7: Cruzamento Ordenado (Ordered Crossover - OX)
O operador OX foi projetado para problemas baseados em permutação:
1. Sorteiam-se dois pontos de corte no Pai 1;
2. A subsequência entre os cortes é copiada diretamente para o Filho nas mesmas posições;
3. As posições restantes são preenchidas com as cidades do Pai 2 na ordem em que aparecem, ignorando as já inseridas.
Demonstração passo a passo:


In [ ]:
def crossover_ordenado_ox_didatico(pai1, pai2, c1, c2):
    tam = len(pai1)
    filho = [None] * tam
    
    # 1. Copia o segmento do Pai 1
    filho[c1:c2] = pai1[c1:c2]
    genes_presentes = set(pai1[c1:c2])
    
    print(f"Pai 1:                     {[nomes_cidades[c] for c in pai1]}")
    print(f"Pai 2:                     {[nomes_cidades[c] for c in pai2]}")
    print(f"Pontos de corte sorteados: índices {c1} até {c2-1}")
    print(f"Segmento herdado do Pai 1: {[nomes_cidades[c] for c in pai1[c1:c2]]}")
    
    # 2. Preenche com a ordem relativa do Pai 2
    ptr_p2 = 0
    for i in range(tam):
        if filho[i] is None:
            while pai2[ptr_p2] in genes_presentes:
                ptr_p2 += 1
            filho[i] = pai2[ptr_p2]
            genes_presentes.add(pai2[ptr_p2])
            
    print(f"Filho Gerado (OX Válido):  {[nomes_cidades[c] for c in filho]}")
    assert len(set(filho)) == tam, "Erro: cidades repetidas ou ausentes no filho!"
    return filho

p1 = [0, 1, 2, 3, 4] # A, B, C, D, E
p2 = [4, 2, 0, 1, 3] # E, C, A, B, D
filho_demonstracao = crossover_ordenado_ox_didatico(p1, p2, c1=1, c2=4)



### Algoritmo Completo: Algoritmo Genético
Integração dos operadores: população inicial, aptidão, seleção por torneio, cruzamento OX, mutação swap e elitismo.


In [ ]:
def crossover_ordenado_ox(pai1, pai2, rng):
    tam = len(pai1)
    filho = [None] * tam
    c1, c2 = sorted(rng.sample(range(tam), 2))
    filho[c1:c2] = pai1[c1:c2]
    genes_presentes = set(pai1[c1:c2])
    ptr_p2 = 0
    for i in range(tam):
        if filho[i] is None:
            while pai2[ptr_p2] in genes_presentes:
                ptr_p2 += 1
            filho[i] = pai2[ptr_p2]
            genes_presentes.add(pai2[ptr_p2])
    return filho

def mutacao_swap(ind, taxa_mutacao, rng):
    if rng.random() < taxa_mutacao:
        n = len(ind)
        i, j = rng.sample(range(n), 2)
        ind[i], ind[j] = ind[j], ind[i]
    return ind

def algoritmo_genetico(cidades, matriz_dist, tam_pop=50, num_geracoes=150, taxa_crossover=0.9, taxa_mutacao=0.05, seed=42):
    rng = random.Random(seed)
    n = len(cidades)
    
    populacao = criar_populacao_inicial(tam_pop, n, rng)
    custos = [calcular_custo_total(ind, matriz_dist) for ind in populacao]
    best_idx = int(np.argmin(custos))
    melhor_solucao = list(populacao[best_idx])
    melhor_custo = custos[best_idx]
    
    historico_custos = [melhor_custo]
    
    for gen in range(num_geracoes):
        nova_populacao = [list(melhor_solucao)] # Elitismo: preserva o melhor individuo historico
        while len(nova_populacao) < tam_pop:
            t1 = rng.sample(range(len(populacao)), 3)
            p1 = populacao[min(t1, key=lambda idx: custos[idx])]
            t2 = rng.sample(range(len(populacao)), 3)
            p2 = populacao[min(t2, key=lambda idx: custos[idx])]
            
            if rng.random() < taxa_crossover:
                filho = crossover_ordenado_ox(p1, p2, rng)
            else:
                filho = list(p1)
                
            filho = mutacao_swap(filho, taxa_mutacao, rng)
            nova_populacao.append(filho)
            
        populacao = nova_populacao
        custos = [calcular_custo_total(ind, matriz_dist) for ind in populacao]
        cur_min_idx = int(np.argmin(custos))
        if custos[cur_min_idx] < melhor_custo:
            melhor_custo = custos[cur_min_idx]
            melhor_solucao = list(populacao[cur_min_idx])
            
        historico_custos.append(melhor_custo)
        
    return melhor_solucao, melhor_custo, historico_custos

# Executando o AG no exemplo brinquedo de 5 cidades
melhor_rota_ag, melhor_custo_ag, hist_ag = algoritmo_genetico(
    cidades_brinquedo, matriz_brinquedo, tam_pop=40, num_geracoes=100, taxa_crossover=0.9, taxa_mutacao=0.05, seed=42
)
print("Resultado do Algoritmo Genético no Problema Brinquedo:")
print(f"  Melhor Rota Encontrada: {formatar_rota(melhor_rota_ag, nomes_cidades)}")
print(f"  Custo Mínimo: {melhor_custo_ag:.2f}")
print(f"  Total de Gerações: {len(hist_ag)}")



---
## Parte 5: Bateria Experimental Comparativa

Execução dos experimentos comparativos nas três instâncias euclidianas:
- **20 cidades**
- **50 cidades**
- **100 cidades**

Ao final, geram-se a tabela de desempenho consolidada e os gráficos comparativos de rotas e convergência.


In [ ]:
def gerar_cidades_aleatorias(n, seed=42):
    rng = random.Random(seed)
    return [(rng.randint(20, 980), rng.randint(20, 980)) for _ in range(n)]

escalas = [20, 50, 100]
resultados_comparativos = []
dados_vis_20 = {}

print("Executando Bateria Comparativa (TS vs. AG)...")
for n in escalas:
    cidades = gerar_cidades_aleatorias(n, seed=42)
    matriz = gerar_matriz_distancias(cidades)
    
    # Executa Têmpera Simulada
    t0 = time.time()
    rota_ts, custo_ts, hist_ts = tempera_simulada(cidades, matriz, temp_inicial=10000, temp_min=0.1, taxa=0.995, seed=42)
    tempo_ts = time.time() - t0
    
    # Executa Algoritmo Genético
    num_ger = 300 if n == 20 else (600 if n == 50 else 1000)
    t0 = time.time()
    rota_ag, custo_ag, hist_ag = algoritmo_genetico(cidades, matriz, tam_pop=100, num_geracoes=num_ger, taxa_crossover=0.9, taxa_mutacao=0.02, seed=42)
    tempo_ag = time.time() - t0
    
    vantagem_ag = ((custo_ts - custo_ag) / custo_ts) * 100.0
    
    resultados_comparativos.append({
        "Nº Cidades": n,
        "Custo TS": round(custo_ts, 2),
        "Tempo TS (s)": round(tempo_ts, 3),
        "Custo AG": round(custo_ag, 2),
        "Tempo AG (s)": round(tempo_ag, 3),
        "Vantagem AG (%)": f"{vantagem_ag:+.2f}%"
    })
    
    if n == 20:
        dados_vis_20 = {
            "cidades": cidades,
            "rota_ts": rota_ts, "custo_ts": custo_ts, "hist_ts": hist_ts,
            "rota_ag": rota_ag, "custo_ag": custo_ag, "hist_ag": hist_ag
        }

df_comparativo = pd.DataFrame(resultados_comparativos)
print("\nTabela Comparativa de Desempenho (TS vs. AG):")
display(df_comparativo)



### Visualização Lado a Lado: Rotas Finais e Curva de Convergência (20 Cidades)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2), dpi=150)
cidades_plot = dados_vis_20["cidades"]

def desenhar_rota(ax, titulo, rota, cidades, cor):
    x = [cidades[i][0] for i in rota] + [cidades[rota[0]][0]]
    y = [cidades[i][1] for i in rota] + [cidades[rota[0]][1]]
    ax.plot(x, y, 'o-', color=cor, linewidth=1.6, markersize=5.5)
    ax.plot(x[0], y[0], 'r*', markersize=14, label='Início / Retorno')
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('Coordenada X', fontsize=9.5)
    ax.set_ylabel('Coordenada Y', fontsize=9.5)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=8.5)

desenhar_rota(axes[0], f'Têmpera Simulada (Custo: {dados_vis_20["custo_ts"]:.2f})', dados_vis_20["rota_ts"], cidades_plot, '#1f77b4')
desenhar_rota(axes[1], f'Algoritmo Genético (Custo: {dados_vis_20["custo_ag"]:.2f})', dados_vis_20["rota_ag"], cidades_plot, '#2ca02c')

# Convergência
axes[2].plot(dados_vis_20["hist_ts"], label='Têmpera Simulada (TS)', color='#1f77b4', linewidth=1.8)
axes[2].plot(dados_vis_20["hist_ag"], label='Algoritmo Genético (AG)', color='#2ca02c', linewidth=1.8)
axes[2].set_title('Convergência do Menor Custo', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Passos de Busca / Gerações', fontsize=9.5)
axes[2].set_ylabel('Custo da Melhor Rota', fontsize=9.5)
axes[2].grid(True, linestyle='--', alpha=0.5)
axes[2].legend(loc='upper right', fontsize=9)

fig.suptitle('Figura Comparativa: Geometria das Rotas e Curvas de Convergência (20 Cidades)', fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()



---
## Parte 6: Revisão Conceitual Rápida

Síntese dos conceitos centrais para revisão da disciplina de Sistemas Inteligentes.

### 1. Parâmetros e Métricas de Comparação:
1. **$n$ (Número de Cidades):** Dimensão da instância. O espaço de rotas distintas tem tamanho $(n-1)! / 2$.
2. **$P$ (Tamanho da População):** Quantidade de soluções candidatas mantidas em memória ($P=1$ na TS; $P \ge 50$ no AG).
3. **$K$ (Iterações ou Gerações):** Número de ciclos executados pelo algoritmo.
4. **$E$ (Avaliações da Função de Custo / Aptidão):** Número total de vezes que uma rota foi avaliada. No AG, $E \approx P \times K$. Na TS com delta em $O(1)$, $E \approx K$. A métrica correta para comparar o orçamento de busca entre os algoritmos é o total de avaliações $E$.

---

### 2. Quadro Comparativo dos Métodos

| Característica | Busca Informada ($A^*$) | Têmpera Simulada (TS) | Algoritmo Genético (AG) |
| :--- | :--- | :--- | :--- |
| **Formulação** | Árvore de estados parciais | Trajetória única em estados completos | População de estados completos |
| **Memória RAM** | $O(b^d)$ — Exponencial | $O(n)$ — Linear | $O(P \cdot n)$ — Linear na população |
| **Custo por Passo** | Cresce com a fronteira aberta | $O(1)$ com delta local | $O(P \cdot n)$ (torneio, crossover e mutação) |
| **Tratamento de Mínimo Local** | Exploração exaustiva | Critério de Metropolis ($e^{-\Delta E / T}$) | Diversidade da população e mutação |
| **Limitação Principal** | Esgotamento de RAM para $n \ge 15$ | Resfriamento rápido leva a congelamento prematuro | Perda de diversidade pode levar à estagnação |

---

### 3. Questões Conceituais Comentadas:

#### **P1: Por que o critério de Metropolis aceita soluções piores no início da busca e quase nenhuma ao final?**
> **Resposta:** No início, a temperatura $T$ é alta, resultando em $-\Delta E / T \approx 0$ e $e^{-\Delta E / T} \approx 1$. Isso permite ao algoritmo explorar o espaço de estados de maneira ampla (*exploration*), escapando de bacias de atração de mínimos locais. Conforme $T \to 0$, a probabilidade decresce, restringindo a busca a movimentos de melhoria estrita (*exploitation*).

#### **P2: Qual a consequência de uma taxa de resfriamento $\alpha$ excessivamente agressiva na Têmpera Simulada?**
> **Resposta:** Ocorre o congelamento prematuro (*quenching*). A temperatura cai antes que o algoritmo explore suficientemente o espaço de soluções, aprisionando a busca em um mínimo local de custo elevado.

#### **P3: Por que o operador de cruzamento simples (1-point crossover) é inadequado para o PCV?**
> **Resposta:** O estado no PCV é uma permutação. Cortar e recombinar dois vetores em um único ponto resulta em cidades duplicadas e cidades omitidas, gerando rotas inválidas. São necessários operadores específicos para permutações, como o Crossover Ordenado (OX) ou PMX.

#### **P4: Qual o impacto de taxas de mutação extremas (0% ou excessivamente altas) no AG?**
> **Resposta:** Uma taxa de mutação muito elevada aproxima o algoritmo de uma busca aleatória, desfazendo blocos de sub-rotas eficientes formados pelo cruzamento. Por outro lado, taxa nula ou muito baixa favorece a perda precoce de diversidade, fazendo a população convergir para cópias idênticas presas em um mesmo mínimo local.

#### **P5: Qual o papel do Elitismo na convergência do Algoritmo Genético?**
> **Resposta:** Como os operadores de cruzamento e mutação são estocásticos, a melhor rota encontrada até determinada geração pode ser modificada ou perdida na geração seguinte. O elitismo copia a melhor solução diretamente para a próxima geração, assegurando que o melhor custo global seja monotonamente não-crescente.

#### **P6: Por que a Têmpera Simulada deve rastrear a melhor rota global visitada (*best-so-far*) em vez de apenas a última rota?**
> **Resposta:** Como a Têmpera Simulada aceita movimentos de piora probabilisticamente ao longo de toda a execução, a rota no término do algoritmo pode ser uma perturbação com custo superior a uma boa solução encontrada em iterações anteriores. Manter o registro da melhor rota global assegura que a melhor solução visitada não seja descartada.
